In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import re
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
arrayPath = Path("/home/philipm/Desktop/proc")
arrayPath.is_dir()

True

In [3]:
outputPath = Path("/home/philipm/Desktop")
outputPath.is_dir()

True

In [4]:
data = pd.DataFrame(columns=["grviIn"])

red = 1123.904772
green = 1020.232214
blue = 845.873822

files = sorted (
    arrayPath.glob("proc_*.npy"),
    key=lambda p: int(p.stem.split("_")[1])
)

In [5]:
for i in range (0, len(files)):
    blocks = np.load(str(files[i]))
    
    for j in range (0, len(blocks)):
        img = blocks[j]

        # Extract RGB channels
        R = img[:, :, 0].astype(np.float64)
        G = img[:, :, 1].astype(np.float64)

        # --------------------------------------------------
        # 1. Create shared vegetation mask
        # --------------------------------------------------

        if img.shape[2] >= 4:
            mask = img[:, :, 3] > 0
        else:
            mask = (R > 0) & (G > 0)

        mask &= np.isfinite(R) & np.isfinite(G)

        # --------------------------------------------------
        # 2. Calculate reflectance
        # --------------------------------------------------

        r_ref = (R * np.pi) / red
        g_ref = (G * np.pi) / green

        # --------------------------------------------------
        # 3. Calculate pixel-wise GRVI
        # --------------------------------------------------

        denominator = g_ref + r_ref

        valid = (
            mask
            & (denominator > 1e-12)
        )

        if not np.any(valid):
            grvi_mean = np.nan
            grvi_std = np.nan

        else:

            grvi_pixels = (
                (g_ref[valid] - r_ref[valid])
                / denominator[valid]
            )

            grvi_mean = np.mean(grvi_pixels)
            grvi_std = np.std(grvi_pixels)

        data.loc[len(data)] = [grvi_mean]


In [6]:
data.to_csv(str(outputPath) + r"/data.csv", index=False)

In [7]:
data

,grviIn
0,0.178121
1,0.175688
2,0.176207
3,0.173518
4,0.165226
...,...
5955,0.165796
5956,0.166375
5957,0.171950
5958,0.177613


In [8]:
gt = pd.read_csv("/media/philipm/FOUR/ThesisData/groundtruthGRVI-front.csv")

In [9]:
gt

,grviOut
0,0.20
1,0.20
2,0.20
3,0.20
4,0.20
...,...
5955,0.34
5956,0.34
5957,0.34
5958,0.34


In [10]:
data["grviOut"] = gt["grviOut"].iloc[:len(data)]
data

,grviIn,grviOut
0,0.178121,0.20
1,0.175688,0.20
2,0.176207,0.20
3,0.173518,0.20
4,0.165226,0.20
...,...,...
5955,0.165796,0.34
5956,0.166375,0.34
5957,0.171950,0.34
5958,0.177613,0.34


In [11]:
data.groupby('grviOut')['grviIn'].mean()

grviOut
0.08    0.170214
0.10    0.155265
0.12    0.172147
0.14    0.139961
0.16    0.141849
0.18    0.156711
0.20    0.155860
0.22    0.156176
0.24    0.147000
0.26    0.156118
0.28    0.153916
0.30    0.149232
0.32    0.152726
0.34    0.149000
0.36    0.155413
0.38    0.151744
Name: grviIn, dtype: float64